In [1]:
import os
import torch
import torchvision
from torch import nn
from d2l import torch as d2l

整理数据集 -> 定义图像增广 -> 读取数据集 -> 定义训练模型 -> 定义损失函数 -> 定义训练函数 -> 开始训练模型

In [ ]:
# 整理数据集
def reorg_dog_data(data_dir, valid_ratio):
    labels = d2l.read_csv_labels(os.path.join(data_dir, 'labels.csv'))
    d2l.reorg_train_valid(data_dir, labels, valid_ratio)  # 分别读出训练图，训练标签，和验证集比例
    d2l.reorg_test(data_dir)

batch_size = 32 
valid_ratio = 0.1
reorg_dog_data(data_dir, valid_ratio)

基础四件套（几乎所有任务都加）

- RandomResizedCrop(224)   # 缩放 + 裁剪 → 模拟不同距离和视角
- RandomHorizontalFlip()   # 水平翻转 → 左右不变性
- ToTensor() # 转为张量
- Normalize(mean, std)     # 数值归一化

ColorJitter	光照变化大的场景（户外、手机拍照）或 颜色是关键特征（医疗染色图、水果成熟度）

Normalize中的参数数据集一般会给，可以搜这个数据集的均值和标准差  
imgs = torch.stack([img for img, _ in train_data], dim=0)  # (N, 3, 32, 32)  
mean = imgs.mean(dim=[0, 2, 3])   # 每个通道的均值  
std = imgs.std(dim=[0, 2, 3])     # 每个通道的标准差  

In [ ]:
# 图像增广
transform_train = torchvision.transforms.Compose([
    torchvision.transforms.RandomResizedCrop(
        224, scale=(0.08, 1.0), ratio=(3.0 / 4.0, 4.0 / 3.0),
    ),
    torchvision.transforms.RandomHorizontalFlip(),
    torchvision.transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

transform_test = torchvision.transforms.Compose([
    torchvision.transforms.Resize(256),
    torchvision.transforms.CenterCrop(224),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [ ]:
# 读取数据集
# os.path.join就是组装路径 os.path.join(a, b, c) = ./a/b/c
train_ds, train_valid_ds = [torchvision.datasets.ImageFolder(
    os.path.join(data_dir, 'train_valid_test', folder),
    transform=transform_train) for folder in ['train', 'train_valid']]

valid_ds, test_ds = [torchvision.datasets.ImageFolder(
    os.path.join(data_dir, 'train_valid_test', folder),
    transform=transform_test) for folder in ['valid', 'test']]

train_iter, train_valid_iter = [torch.utils.data.DataLoader(
    dataset, batch_size, shuffle=True, drop_last=True)
    for dataset in (train_ds, train_valid_ds)
]

valid_iter = torch.utils.data.DataLoader(valid_ds, batch_size, shuffle=False, drop_last=False)

test_iter = torch.utils.data.DataLoader(test_ds, batch_size, shuffle=False, drop_last=False)

In [ ]:
# 微调预训练模型
def get_net(devices):  
    finetune_net = nn.Sequential()  # 创建一个容器
    finetune_net.features = torchvision.models.resnet34(pretrained=True)  # 把 ImageNet 预训练的 ResNet-34 塞进去，叫 features
    finetune_net.output_new = nn.Sequential(nn.Linear(1000, 256), nn.ReLU(), nn.Linear(256, 120))  # 新建一个分类头 output_new，替代原来的 fc
    finetune_net = finetune_net.to(devices[0])  # 整个模型搬到 GPU
    for param in finetune_net.features.parameters():  # 只训练分类头
        param.requires_grad = False
    return finetune_net

In [ ]:
loss = nn.CrossEntropyLoss(reduction='none')  # 定义损失函数

def evaluate_loss(data_iter, net, devices):
    l_sum, n = 0.0, 0
    for features, labels in data_iter:
        features, labels = features.to(devices[0]), labels.to(devices[0])  # 把数据和标签从 CPU 内存搬到 GPU 显存。
        outputs = net(features)  # 得到预测结果
        l = loss(outputs, labels)
        l_sum += l.sum()  # 损失值总量
        n += labels.numel()  # 标签总量
    return l_sum / n

In [ ]:
def train(net, train_iter, valid_iter, num_epochs, lr, wd, devices, lr_period, lr_decay):
    net = nn.DataParallel(net, device_ids=devices).to(devices[0])  # 同时gpu并行训练
    trainer = torch.optim.SGD((param for param in net.parameters() if param.requires_grad == True), lr=lr, momentum=0.9, weight_decay=wd)  # 随机梯度下降更新数据
    scheduler = torch.optim.lr_scheduler.StepLR(trainer, lr_period, lr_decay)
    num_batches, timer = len(train_iter), d2l.Timer()
    legend = ['train loss']
    if valid_iter is not None:
        legend.append('valid loss')
    animator = d2l.Animator(xlabel='epoch', xlim=[1, num_epochs], legend=legend)

    for epoch in range(num_epochs):
        metric = d2l.Accumulator(2)
        for i, (features, labels) in enumerate(train_iter):
            timer.start()
            features, labels = features.to(devices[0]), labels.to(devices[0])
            trainer.zero_grad()
            output = net(features)
            l = loss(output, labels).sum()
            l.backward()  # 反向传播
            trainer.step()  # 更新参数
            metric.add(l, labels.shape[0])  # 批次总loss  这个批次的样本数 用于计算平均loss
            timer.stop()
            '''
            在训练过程中每隔 1/5 个 epoch 更新一次曲线
            (i + 1) % (num_batches // 5) == 0    # 每跑完 20% 的 batch 时
            i == num_batches - 1  # 或最后一个 batch 时（保证至少画一次）
            '''
            if (i + 1) % (num_batches // 5) == 0 or i == num_batches - 1:
                animator.add(epoch + (i + 1) / num_batches,
                             (metric[0] / metric[1], None))
        measures = f'train loss {metric[0] / metric[1] :.3f}'
        # 每个 epoch 结束时：算验证 loss + 更新学习率
        if valid_iter is not None:
            valid_loss = evaluate_loss(valid_iter, net, devices)  # 用验证集算一次 loss
            animator.add(epoch + 1, (None, valid_loss))  # 在曲线上描一个验证 loss 点
        scheduler.step()  # 学习率按阶梯下降
    # 拼上最后一轮的验证 loss
    if valid_iter is not None:
        measures += f', valid loss {valid_loss:.3f}'
    print(measures + f'\n{metric[2] * num_epochs} samples processed')

In [ ]:
# 训练模型
devices, num_epochs, lr, wd = d2l.try_all_gpus(), 10, 1e-4, 1e-4
lr_period, lr_decay, net = 2, 0.9, get_net(devices)
train(net, train_iter, valid_iter, num_epochs, lr, wd, devices, lr_period, lr_decay)

In [ ]:
# 测试
preds = []
for data, label in test_iter:
    output = torch.nn.functional.softmax(net(data.to(devices[0])), dim=1)  # 对每个样本在类别维度上做 softmax。
    preds.extend(output.cpu().detach().numpy())

ids = sorted(os.listdir(os.path.join(data_dir, 'train_valid_iter', 'test')))